Импортируем библиотеки

In [ ]:
!pip install scikeras

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GRU, Dropout, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

## Подготовка данных

Загружаем датасет

In [ ]:
uploaded = files.upload()

In [ ]:
data = pd.read_csv('emotions.csv')

In [ ]:
data

Выделение признаков и меток

In [ ]:
fft_columns = [col for col in data.columns if col.startswith("fft")]
data_fft = data[fft_columns]

labels = data['label']

In [ ]:
data_fft.head()

In [ ]:
labels.head()

Кодирование меток классов

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

Формирование массивов признаков и меток

In [ ]:
X = data_fft.values
y = np.array(y)

Нормализация признаков

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

Разделение данных на обучающую и тестовую выборки

In [ ]:
X_flat = X

# Разделяем данные: 80% — обучение, 20% — тест
X_tr_flat, X_te_flat, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
def to_gru(x_flat):
    # Делим признаки на две половины
    mid = x_flat.shape[1] // 2
    a = x_flat[:, :mid]    # первая половина
    b = x_flat[:, mid:]    # вторая половина
    return np.stack((a, b), axis=-1)

In [ ]:
X_train = to_gru(X_tr_flat)
X_test  = to_gru(X_te_flat)

## Настройка методов отбора признаков

Теперь мы проведём **сравнительный анализ различных методов отбора признаков**.  
Для каждого метода из словаря selectors мы переберём заданные параметры и оценим качество модели с выбранными признаками, используя кросс-валидацию.

In [ ]:
selectors = {
    'SelectKBest': {
        'class': SelectKBest,
        'params': {'score_func': f_classif},
        'grid': {'k': [50, 100, 500, X.shape[1]]}  # число лучших признаков
    },
    'SelectFromModel_RF': {
        'class': SelectFromModel,
        'params': {
            'estimator': RandomForestClassifier(n_estimators=200, random_state=42)
        },
        'grid': {'threshold': ['median']}  # отбираем признаки выше медианы важности
    },
    'L1_LinearSVC': {
        'class': SelectFromModel,
        'params': {
            'estimator': LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
        },
        'grid': {'threshold': [None]}  # оставляем признаки с ненулевыми весами
    }
}

In [ ]:
best_acc = 0.0
best_name = None
best_selector = None

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
# Проходим по каждому способу отбора признаков
for name, spec in selectors.items():
    Cls         = spec['class']
    base_kwargs = spec['params']
    for param, values in spec['grid'].items():
        for v in values:
            # Создаём селектор с конкретным параметром
            sel_kwargs = base_kwargs.copy()
            sel_kwargs[param] = v
            sel = Cls(**sel_kwargs)

            # Отбираем признаки на обучающих данных
            X_tr_sel = sel.fit_transform(X_tr_flat, y_train)
            X_va_sel = sel.transform(X_te_flat)

            rf = RandomForestClassifier(n_estimators=100, random_state=42)
            # Оцениваем модель по кросс-валидации
            scores = []
            for train_idx, val_idx in cv.split(X_tr_sel, y_train):
                rf.fit(X_tr_sel[train_idx], y_train[train_idx])
                scores.append(rf.score(X_tr_sel[val_idx], y_train[val_idx]))
            mean_cv = np.mean(scores)

            # Считаем число признаков
            n_feats = X_tr_sel.shape[1]

            print(f"{name}: {param}={v} → CV acc={mean_cv:.2f}, n_feats={n_feats}")

            # Обновим лучший результат, если текущий лучше
            if mean_cv > best_acc:
                best_acc      = mean_cv
                best_name     = name
                best_selector = sel

In [ ]:
print(f"\nЛучший метод: {best_name}, CV accuracy={best_acc:.2f}")

После перебора всех методов и параметров с использованием кросс-валидации был выбран наиболее эффективный способ отбора признаков: L1_LinearSVC.
Это означает, что модель LinearSVC с L1-регуляризацией смогла эффективно обнулить менее важные признаки и оставить только значимые.

Теперь мы можем применить best_selector для сокращения признаков в обучающей и тестовой выборках перед подачей в модель GRU.

In [ ]:
best_selector.fit(X_tr_flat, y_train)
X_sel_flat = best_selector.transform(X_tr_flat)
X_sel_count = X_sel_flat.shape[1]

print(f"   Оставлено признаков: {X_sel_count}")

Осталось 196 признаков, которые были признаны наиболее значимыми на основе весов модели LinearSVC с L1-регуляризацией.

In [ ]:
X_tr_sel_flat = best_selector.transform(X_tr_flat)
X_te_sel_flat = best_selector.transform(X_te_flat)
X_train_sel = to_gru(X_tr_sel_flat)
X_test_sel  = to_gru(X_te_sel_flat)

# Настройка и обучение GRU

Построение и конфигурация модели GRU

In [ ]:
param_grid = {
    'units':   [32, 64],     # количество нейронов в GRU
    'dropout': [0.2, 0.5],   # уровень Dropout-регуляризации
    'lr':      [1e-3, 1e-4], # скорость обучения (learning rate)
}
fixed = {
    'features':    2,     # количество каналов на каждый шаг
    'dense_units': 32,    # размер промежуточного Dense-слоя
    'batch_size':  32,
    'epochs':      50,
    'patience':     5     # EarlyStopping — количество эпох без улучшения
}

Для удобства повторного использования и автоматического перебора параметров  
создадим функцию `make_model`, которая будет возвращать готовую модель GRU  
с заданными значениями `units`, `dropout` и `learning rate`.

GRU(units): рекуррентный слой для обработки последовательных признаков.

Dropout(dropout): регуляризация для борьбы с переобучением.

Dense(fixed['dense_units']): промежуточный слой с активацией ReLU.

Dense(len(label_encoder.classes_)): выходной слой с softmax для классификации на 3 эмоции.

Adam(learning_rate=lr): оптимизатор с переданной скоростью обучения.

In [ ]:
def make_model(units, dropout, lr):
    m = Sequential([
        GRU(units, input_shape=(fixed['timesteps'], fixed['features'])),
        Dropout(dropout),
        Dense(fixed['dense_units'], activation='relu'),
        Dense(len(label_encoder.classes_), activation='softmax')
    ])
    m.compile(
        optimizer=Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return m

In [ ]:
# Эта функция обучает модель с заданными гиперпараметрами и возвращает максимальную точность на валидации
def eval_params(params):
    m = make_model(**params)
    es = EarlyStopping(monitor='val_accuracy',
                       patience=fixed['patience'],
                       restore_best_weights=True)
    hist = m.fit(
        X_train_sel, y_train,
        validation_data=(X_test_sel, y_test),
        epochs=fixed['epochs'],
        batch_size=fixed['batch_size'],
        callbacks=[es],
        verbose=0
    )
    return max(hist.history['val_accuracy'])

In [ ]:
best_params = {}
defaults = {k:v[0] for k,v in param_grid.items()}

Поэтапный подбор гиперпараметров GRU

In [ ]:
for key in ['units', 'dropout', 'lr']:
    best_acc = 0
    for val in param_grid[key]:
        params = defaults.copy()
        params[key] = val
        acc = eval_params(params)
        print(f"Testing {key}={val} → val_acc={acc:.2f}")
        if acc > best_acc:
            best_acc = acc
            best_params[key] = val
    defaults[key] = best_params[key]
    print(f"→ Best {key}: {best_params[key]} (val_acc={best_acc:.2f})\n")

Меньшее количество нейронов оказалось более устойчивым и показало лучшую валидационную точность.

Меньшее значение Dropout (0.2) дало лучшее качество,
возможно, из-за отсутствия переобучения благодаря отбору признаков.

Больший learning_rate обеспечил быструю и стабильную сходимость,
в то время как слишком маленькое значение затруднило обучение.

Финальные параметры

In [ ]:
print(f"nBest params:", best_params)

In [ ]:
u = best_params['units']
d = best_params['dropout']
l = best_params['lr']

Финальная модель GRU с подобранными параметрами

In [ ]:
# Создание финальной модели  с подобранными параметрами
best_model = Sequential([
    GRU(u, input_shape=(196, 2), return_sequences=False),  # GRU-слой
    Dropout(d),                                             # Dropout-регуляризация для предотвращения переобучения
    Dense(32, activation='relu'),                           # Полносвязный скрытый слой
    Dense(len(label_encoder.classes_), activation='softmax')  # Выходной слой с softmax для многоклассовой классификации
])

# Компиляция модели
best_model.compile(
    optimizer=Adam(learning_rate=l),                        # Адам-оптимизатор с подобранным learning rate
    loss='sparse_categorical_crossentropy',                 # Функция потерь для целочисленных меток
    metrics=['accuracy']                                    # Метрика — точность
)

# EarlyStopping: остановка обучения при отсутствии улучшения val_accuracy
es = EarlyStopping(
    monitor='val_accuracy',     # Мониторим валидационную точность
    patience=5,                 # Кол-во эпох без улучшения до остановки
    restore_best_weights=True  # Восстановить лучшие веса после остановки
)

Обучение модели на выбранных признаках

In [ ]:
history = best_model.fit(
    X_train_sel, y_train,
    validation_data=(X_test_sel, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[es],
    verbose=1
)

Оценка качества модели на тестовой выборке

In [ ]:
# Получение вероятностей предсказаний на тестовой выборке
y_pred_proba = best_model.predict(X_test_sel)

# Преобразование вероятностей в предсказанные классы
y_pred = np.argmax(y_pred_proba, axis=1)

# Вычисление F1-метрики
f1 = f1_score(y_test, y_pred, average='weighted')

# Построение матрицы ошибок (confusion matrix)
cm = confusion_matrix(y_test, y_pred)

In [ ]:
print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1 Score: {f1:.4f}\n")

Модель успешно обучилась и демонстрирует хорошее качество предсказаний

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

NEGATIVE: Отличный recall (0.99) — модель почти всегда правильно распознаёт негативные эмоции, но precision чуть ниже (модель иногда ошибочно даёт негативный класс).

NEUTRAL: Хорошие значения — модель чётко распознаёт нейтральные состояния.

POSITIVE: Немного хуже recall (0.89), значит часть положительных примеров модель путает с другими (возможно с "NEGATIVE").

Визуализация истории обучения: потери и точность по эпохам

In [ ]:
plt.figure(figsize=(12,5))

In [ ]:
# График функции потерь (Loss)
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

In [ ]:
# График точности (Accuracy)
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

Визуализация матрицы ошибок (Confusion Matrix)

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Настройка и обучение CNN+GRU

Модель CNN + GRU: объединение свёрточных и рекуррентных слоёв

Для улучшения качества классификации мы реализуем гибридную архитектуру,  
сочетающую **свёрточную нейросеть (CNN)** и **GRU**.  
CNN поможет выявить локальные паттерны, а GRU — учесть временную зависимость.

In [ ]:
param_grid_cnn = {
    'filters':     [32, 64],
    'kernel_size': [3, 5],
    'gru_units':   [32, 64],
}

fixed_cnn = {
    'features':      2,
    'pool_size':     2,
    'dropout':     0.3,
    'dense_units': 32,
    'batch_size':  32,
    'epochs':      50,
    'patience':     5,
    'lr':        1e-3,
}

In [ ]:
# Функция создания модели
def make_cnn_gru(filters, kernel_size, gru_units):
    model = Sequential([
        # 1D-свёртка: извлечение локальных признаков
        Conv1D(filters, kernel_size, activation='relu',
               input_shape=(fixed_cnn['timesteps'], fixed_cnn['features'])),

        # Уменьшение размерности выхода свёртки
        MaxPooling1D(fixed_cnn['pool_size']),

        # Dropout для регуляризации
        Dropout(fixed_cnn['dropout']),

        # GRU-слой: захватывает временные зависимости в выходе свёртки
        GRU(gru_units),

        # Ещё один Dropout
        Dropout(fixed_cnn['dropout']),

        # Полносвязный слой + выходной softmax
        Dense(fixed_cnn['dense_units'], activation='relu'),
        Dense(len(label_encoder.classes_), activation='softmax')
    ])

    # Компиляция модели
    model.compile(
        optimizer=Adam(learning_rate=fixed_cnn['lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
# Функция оценки параметров
def eval_cnn(params):
    # Создание модели по переданным параметрам
    m = make_cnn_gru(**params)

    # EarlyStopping — остановка обучения при отсутствии улучшения
    es = EarlyStopping(
        monitor='val_accuracy',
        patience=fixed_cnn['patience'],
        restore_best_weights=True
    )

    # Обучение модели
    hist = m.fit(
        X_train_sel, y_train,                  # обучающая выборка
        validation_data=(X_test_sel, y_test), # валидационная выборка
        epochs=fixed_cnn['epochs'],
        batch_size=fixed_cnn['batch_size'],
        callbacks=[es],
        verbose=0
    )

    # Возвращаем наивысшую достигнутую точность и историю обучения
    return max(hist.history['val_accuracy']), hist

Поэтапный подбор параметров для модели CNN + GRU, на каждом шаге фиксируем лучший найденный параметр и ищем оптимальный среди следующего списка.

In [ ]:
best_cnn = {}
defaults = {k: v[0] for k, v in param_grid_cnn.items()}

for key in ['filters', 'kernel_size', 'gru_units']:
    best_acc = 0.0
    for val in param_grid_cnn[key]:
        params = defaults.copy()
        params[key] = val
        val_acc, _ = eval_cnn(params)
        print(f"Testing {key}={val} → val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_cnn[key] = val
    defaults[key] = best_cnn[key]
    print(f"→ Best {key}: {best_cnn[key]} (val_acc={best_acc:.4f})\n")

32 фильтра показывают лучшую валидационную точность — почти 99.5%. Увеличение количества фильтров до 64 не даёт прироста, а даже снижает точность.

Узкое ядро (3) эффективнее, вероятно, потому что лучше захватывает локальные паттерны в данных ЭЭГ. Широкое ядро теряет точность.

Увеличение числа нейронов GRU до 64 даёт небольшое, но стабильное улучшение — возможно, оно позволяет лучше улавливать временные зависимости в признаках.

Итоговая конфигурация

In [ ]:
print("Optimal CNN+GRU hyperparameters:", best_cnn)

Финальная модель CNN + GRU и обучение

In [ ]:
# Создание модели с лучшими подобранными параметрами
final_model = make_cnn_gru(**best_cnn)

# Callback для остановки обучения при отсутствии улучшения по валидации
es = EarlyStopping(
    monitor='val_accuracy',
    patience=fixed_cnn['patience'],
    restore_best_weights=True
)

# Обучение модели
history_cnn = final_model.fit(
    X_train_sel, y_train,                         # обучающая выборка
    validation_data=(X_test_sel, y_test),        # валидация на тесте
    epochs=fixed_cnn['epochs'],                  # максимальное количество эпох
    batch_size=fixed_cnn['batch_size'],          # размер батча
    callbacks=[es],                              # ранняя остановка
    verbose=1                                    # отображение прогресса
)

Оценка качества финальной модели CNN + GRU

In [ ]:
# Получение вероятностей предсказаний для тестовой выборки
y_pred_proba = final_model.predict(X_test_sel)

# Получение меток классов по максимальной вероятности
y_pred = y_pred_proba.argmax(axis=1)

# Вывод точности (Accuracy)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

# Взвешенный F1 Score (учитывает дисбаланс классов)
print("Test F1 Score:", f1_score(y_test, y_pred, average='weighted'))

# Подробный отчёт по классам
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Test Accuracy: 0.9857 — модель почти не ошибается.

Test F1 Score: 0.9845 — сбалансированное качество предсказаний по всем классам.

NEUTRAL и POSITIVE распознаются почти безошибочно.

NEGATIVE слегка отстаёт по precision (возможны небольшие ложные срабатывания), но всё равно на очень хорошем уровне.

Значения macro avg и weighted avg также указывают на сбалансированное качество по всем классам.

Визуализация истории обучения модели (loss и accuracy)

In [ ]:
plt.figure(figsize=(12, 5))

# Потери (Loss)
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['loss'], label='Train Loss')
plt.plot(history_cnn.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Точность (Accuracy)
plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['accuracy'], label='Train Acc')
plt.plot(history_cnn.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

Построение матрицы ошибок (Confusion Matrix)

In [ ]:
y_pred_proba = final_model.predict(X_test_sel)
y_pred = y_pred_proba.argmax(axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Выводы

В рамках проекта была решена задача автоматической классификации эмоционального состояния человека на основе данных ЭЭГ.

Для подготовки данных была проведена:
*   Нормализация признаков
*   Разделение данных на обучающую и тестовую выборки

Чтобы улучшить обобщающую способность моделей и уменьшить размерность пространства, мы протестировали несколько методов отбора признаков:
* SelectKBest (на основе статистики ANOVA F-test)
* SelectFromModel с использованием RandomForestClassifier
* L1-регуляризованный LinearSVC, отбирающий признаки с ненулевыми коэффициентами

По результатам перекрестной проверки (3-fold StratifiedKFold), лучший результат показал метод L1_LinearSVC, который оставил 196 признаков из более чем 1500. Это позволило значительно сократить входное пространство и ускорить обучение, не потеряв в точности.

Следующим шагом было обучение рекуррентной нейросети, способной обрабатывать временные зависимости. Для этого использовалась архитектура с GRU-слоем:
* Выходной слой softmax предсказывает одну из трех эмоций
* Гиперпараметры (кол-во нейронов, dropout, learning rate) подбирались вручную
* Обучение контролировалось с помощью EarlyStopping

Результаты:
* Accuracy: 0.9461
* F1-мера: 0.9415

Наилучшая точность у нейтрального класса — почти 0.99, модель очень хорошо распознаёт состояния «спокойствия» или «отсутствия выраженной эмоции».

Однако в классе POSITIVE заметна проблема с recall — модель не всегда «узнаёт» позитивные эмоции, и часть таких случаев ошибочно относит к другим категориям.

Модель показала очень достойный результат и вполне пригодна для реальных задач. Она лёгкая, обучается быстро и не требует сложных архитектур. Однако её слабое место — возможное «смешение» позитивных и нейтральных сигналов.

Для улучшения качества классификации была построена гибридная модель CNN + GRU, объединяющая:
* Сверточные слои (Conv1D) — выявляют локальные паттерны в сигналах.
* GRU — фиксируют временные зависимости в преобразованных данных.
* Полносвязные слои и softmax — формируют итоговый прогноз.

Результаты:
* Accuracy: 0.9857
* F1-мера: 0.9844

Все классы предсказываются почти идеально: recall и precision по каждому классу около 0.98–0.99.

Модель уверенно различает даже похожие по структуре эмоции, не теряя точности.

Эта модель продемонстрировала хорошее качество и явно превосходит GRU по всем показателям. Она не только точнее, но и устойчивее — даже сложные сигналы классифицируются корректно.

В реальных условиях, например, при создании систем для эмоциональной адаптации приложений или анализа психоэмоционального состояния, гибридная архитектура CNN+GRU будет наиболее оправданным выбором. Однако для быстрого прототипирования и менее критичных задач можно использовать и GRU, которая даёт очень конкурентоспособные результаты.